# 06 - Use the tuned model as an agent sub-model

Fine-tuning is not a trophy, it is a cheap component. Here the tuned extractor
is one tool the agent calls on its high-frequency step, with a fallback to a
strong model when the cheap one cannot produce valid JSON. That escalation
rate, times the price gap, is your real, defensible saving.

In [ ]:
import sys; sys.path.insert(0, '../src')
from qlora_lab import schema, synth, dataset, predict, evaluate, train, serve, agent

In [ ]:
tuned_client = serve.openai_client(); TUNED = 'ticket'
from openai import OpenAI
strong_client = OpenAI(); STRONG = 'gpt-4o-mini'
msg = 'Hey team, order 55012 arrived smashed and I am furious. Please advise.'
print(agent.extract_ticket_tool(tuned_client, TUNED, msg))

### Cheap-first routing with fallback

In [ ]:
test = dataset.read_jsonl('../data/test.jsonl')[:50]
results = [agent.route_with_fallback(e['message'], tuned_client, TUNED,
                                     strong_client, STRONG) for e in test]
from collections import Counter
print(Counter(r['served_by'] for r in results))
# e.g. {'tuned_small': 47, 'strong_fallback': 3} -> 94% served cheap

That Counter is the slide: most traffic served by the cheap self-hosted model,
a small tail escalated. This is the VortexNet bullet made real - a high-volume
step migrated off the frontier API, with numbers to defend it.